# 3D Arena Reconstruction - Gaussian Splatting

Reconstruct a 3D arena from 93 images using Gaussian Splatting on Google Colab.

## Steps:
1. Upload resized images
2. Run COLMAP for Structure-from-Motion
3. Convert to Gaussian Splatting format
4. Train and extract point cloud

---

In [ ]:
#@title 1. Install Dependencies
%cd /content

# Update and install system packages
!apt-get update
!apt-get install -y colmap wget unzip

# Install Python packages
!pip install torch==2.1.0+cu118 torchvision==0.16.0+cu118 --index-url https://download.pytorch.org/whl/cu118
!pip install plyfile numpy pillow opencv-python-headless tqdm

In [ ]:
#@title 2. Clone 3D Gaussian Splatting Repo
%cd /content

# Clone official repo with submodules
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting
%cd /content/gaussian-splatting

# Install the differentiable rasterizer
!pip install submodules/diff-gaussian-rasterization
!pip install submodules/simple-knn

# Verify installation
!python -c "import diff_gaussian_rasterization; print('diff-gaussian-rasterization OK')"
!python -c "import simple_knn; print('simple-knn OK')"

In [ ]:
#@title 3. Upload Images
from google.colab import files
import os
import zipfile

# Create directories
!mkdir -p /content/gaussian-splatting/input_images
!mkdir -p /content/gaussian-splatting/output

# Upload
print("Upload your resized images as a ZIP file")
uploaded = files.upload()

# Extract
for fname in uploaded.keys():
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as zip_ref:
            zip_ref.extractall('/content/gaussian-splatting/input_images')

# Count images
img_dir = '/content/gaussian-splatting/input_images'
imgs = [f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
print(f"Loaded {len(imgs)} images")

In [ ]:
#@title 4. Run COLMAP (Structure from Motion)
%cd /content/gaussian-splatting

input_path = "input_images"
output_path = "output/sparse"

!mkdir -p {output_path}

# Feature extraction
!colmap feature_extractor \
    --database_path {output_path}/database.db \
    --image_path {input_path}

# Feature matching
!colmap exhaustive_matcher \
    --database_path {output_path}/database.db

# Sparse 3D reconstruction
!colmap mapper \
    --database_path {output_path}/database.db \
    --image_path {input_path} \
    --output_path {output_path}

# Check results
!ls {output_path}/0/

In [ ]:
#@title 5. Convert COLMAP output to Gaussian Splatting format
%cd /content/gaussian-splatting

# Create output directories
!mkdir -p output/train

# Convert
!python convert.py \
    --source_path input_images \
    --model_path output/sparse/0 \
    --output_path output/train

# Check output
!ls output/train/

In [ ]:
#@title 6. Train Gaussian Splatting
%cd /content/gaussian-splatting

# Train - 3000 iterations is good for quick results
# Adjust iterations for better quality
!python train.py \
    -s output/train \
    -i 0 \
    --iterations 3000 \
    --checkpoint 500

In [ ]:
#@title 7. Extract Point Cloud
from plyfile import PlyData, PlyElement
import os
import numpy as np

# Find the point cloud
model_path = "output/train/point_cloud"
iter_path = None
for f in os.listdir(model_path):
    if f.startswith('iteration_'):
        iter_path = os.path.join(model_path, f)
        break

if iter_path:
    print(f"Looking in: {iter_path}")
    !ls {iter_path}
    
    # Find .ply file
    ply_file = None
    for f in os.listdir(iter_path):
        if f.endswith('.ply'):
            ply_file = os.path.join(iter_path, f)
            break
    
    if ply_file:
        # Copy to output
        !cp {ply_file} /content/arena_point_cloud.ply
        print(f"Point cloud saved: /content/arena_point_cloud.ply")
        !ls -lh /content/arena_point_cloud.ply
else:
    print("No iterations found")
    !ls {model_path}

In [ ]:
#@title 8. Download Results
from google.colab import files

# Download point cloud
if os.path.exists('/content/arena_point_cloud.ply'):
    files.download('/content/arena_point_cloud.ply')

# Also download full model
!zip -r /content/3dgs_model.zip output/train
files.download('/content/3dgs_model.zip')

## Usage Tips

- **Images**: Resize to max 1920px width before uploading (use prepare_images.py)
- **Iterations**: More iterations = better quality but longer training
- **VRAM**: Colab T4 has 15GB, should handle 93 images fine
- **Point Cloud**: Open the .ply file in MeshLab or CloudCompare